# ZeroDefect: train the YOLO defect detector on a GPU (M2)

Runs on **Google Colab** (Runtime → Change runtime type → **T4 GPU**) or **Kaggle** (Settings → Accelerator → **GPU T4 x2**, Internet **On**).

What it does:
1. Downloads **Real-IAD** (5 polymer part types, 1024 px, about 8.8 GB) and **PaintDefect**, and converts them into the project's unified format.
2. Builds the YOLO dataset: whole parts are kept together, and the test set stays inside the Real-IAD official test set.
3. Fine-tunes **YOLO26s**, evaluates it on the held-out test set (mAP, per-class P/R/F1, false-reject and escape rates per part) and exports it to **ONNX** for CPU inference.

**Keys:** add `HF_TOKEN` and `ROBOFLOW_API_KEY` as secrets first (Colab: the 🔑 panel on the left, and allow this notebook to access them; Kaggle: Add-ons → Secrets). See `docs/SETUP.md`.

**Time:** about 20 minutes for download and conversion, then training for up to `HOURS` (default 5). Run all cells top to bottom (Colab: Runtime → Run all).

Licences: Real-IAD CC BY-NC-SA 4.0 and Ultralytics AGPL-3.0, both fine for this academic project; PaintDefect CC BY 4.0.

In [ ]:
# Settings
BRANCH = "claude/magical-feynman-bx2o1g"  # repository branch with the detector code
RESOLUTION = "1024"  # Real-IAD copy to download: 512 (2.3 GB) or 1024 (8.8 GB)
IMGSZ = 1024  # training image size
MODEL = "yolo26s.pt"  # yolo26n.pt is faster, yolo26m.pt more accurate
EPOCHS = 100
HOURS = 5  # stop after this many hours, whatever EPOCHS says (sessions time out)
BATCH = -1  # -1 = pick the largest batch that fits the GPU (single GPU only)
RUN = f"polymer_{IMGSZ}_{MODEL.removesuffix('.pt')}"

In [ ]:
# Platform, secrets, code and packages
import os
import subprocess
import sys
from pathlib import Path

ON_KAGGLE = Path("/kaggle").exists()
ON_COLAB = "google.colab" in sys.modules


def secret(name):
    if os.environ.get(name):
        return os.environ[name]
    try:
        if ON_COLAB:
            from google.colab import userdata

            return userdata.get(name)
        if ON_KAGGLE:
            from kaggle_secrets import UserSecretsClient

            return UserSecretsClient().get_secret(name)
    except Exception as e:  # missing secret or no notebook access
        print(f"{name}: not available ({type(e).__name__})")
    return None


for name in ("HF_TOKEN", "ROBOFLOW_API_KEY"):
    value = secret(name)
    if value:
        os.environ[name] = value
    print(f"{name}: {'found' if value else 'MISSING - add it as a secret and re-run this cell'}")

WORK = Path("/kaggle/working") if ON_KAGGLE else Path("/content")
REPO = WORK / "ZeroDefect"
# Datasets are large and need not be saved with the notebook output: keep them out of /kaggle/working.
os.environ["ZERODEFECT_DATA_ROOT"] = "/tmp/zerodefect-data" if ON_KAGGLE else str(WORK / "data")

if not REPO.exists():
    subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            "-b",
            BRANCH,
            "https://github.com/mescorastudios-creator/ZeroDefect.git",
            str(REPO),
        ],
        check=True,
    )
os.chdir(REPO)
# The preinstalled PyTorch (with CUDA) is kept; only the detector packages are added.
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "ultralytics>=8.4",
        "onnx>=1.17",
        "onnxslim>=0.1.82",
        "onnxruntime>=1.20",
    ],
    check=True,
)

In [ ]:
# Check the GPU
import torch

print(
    "GPU:",
    torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE: switch the runtime to a GPU",
)

In [ ]:
# Download and convert the datasets (about 20 minutes; skipped for data already on disk)
!python -m ml.datasets download realiad --resolution {RESOLUTION} --convert
!python -m ml.datasets download paintdefect --convert
!python -m ml.datasets stats

In [ ]:
# Build the YOLO dataset
!python -m ml.detector build --sources realiad paintdefect --name {RUN}

In [ ]:
# Train (progress, losses and mAP per epoch are printed below; plots go to outputs/detector/RUN)
args = f"--data {RUN} --model {MODEL} --epochs {EPOCHS} --time {HOURS} --imgsz {IMGSZ} --batch {BATCH}"
!python -m ml.detector train {args} --device 0 --workers 4 --name {RUN}

In [ ]:
# Evaluate the best weights on the held-out test set
import json

import pandas as pd

WEIGHTS = REPO / "outputs/detector" / RUN / "weights/best.pt"
!python -m ml.detector eval --weights {WEIGHTS} --data {RUN} --split test --imgsz {IMGSZ} --device 0

m = json.loads((WEIGHTS.parent.parent / "eval/metrics_test.json").read_text())
print({k.split("/")[-1]: round(v, 3) for k, v in m["overall"].items()})
display(pd.DataFrame(m["per_class"]).set_index("Class").round(3))
display(pd.DataFrame({"image": m["line"]["image"], **m["line"].get("part", {})}).T)

In [ ]:
# Export to ONNX (CPU inference) and save the run
!python -m ml.detector export --weights {WEIGHTS} --imgsz {IMGSZ}

import shutil

archive = shutil.make_archive(str(WORK / RUN), "zip", REPO / "outputs/detector", RUN)
print("saved", archive)
if ON_COLAB:
    from google.colab import drive, files

    try:
        drive.mount("/content/drive")
        target = Path("/content/drive/MyDrive/ZeroDefect")
        target.mkdir(parents=True, exist_ok=True)
        shutil.copy(archive, target)
        print("copied to Google Drive:", target)
    except Exception as e:
        print("Drive not mounted:", e)
        files.download(archive)
# Kaggle: the zip is in /kaggle/working; use "Save Version" to keep it, or download it from the Output panel.